# Visualization

## Collecting data

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import sys
sys.path.append('../../../../foamalyze/evaluation/')
import process
import collect
import assess

# meaningful tests will fail, if in one-timestep-testing
if assess.is_in_folder():
    assess.writeResult(0)
    sys.exit(0)

# Set style for plots
sns.set_theme(style="whitegrid")

# Set parameter which timeSteps to evaluate
evalStartTime = 0
evalEndTime = float(collect.readEndTime('.'))
evalNthTime = 1 # parameter to evaluate only every n-th timestep
# endTime = collect.readEndTime('.')
# if str(evalEndTime) != endTime:
#     print("Warning: not evaluating at latest timeStep from controlDict.\n"
#             + "\tEvaluating at " + str(evalEndTime)
#             + ", but latest timeStep is " + str(endTime) + ".")

TimeStepFolders = collect.getTimeStepFolders()
TimeStepFolders = TimeStepFolders[1::evalNthTime] # no value in 0/
print(TimeStepFolders)

# create lists of parameters
XFs = []
UFs = []

# read diffusivity and cell-width of parameter studies
for time in TimeStepFolders:
    if float(time) >= evalStartTime and float(time) <= evalEndTime:
        if os.path.isfile(time+'/movingReferenceFrame'):
            # print("File found in " + time)
            Xf = collect.readDictionaryEntry(time,"XF",'/movingReferenceFrame',1,"vector")
            XFs.append(Xf)

            Uf = collect.readDictionaryEntry(time,"UF",'/movingReferenceFrame',1,"vector")
            UFs.append(Uf)
        else:
            print("File not found in " + time)

print(XFs)
print(UFs)

## Plot over time

In [ ]:
def plot(x, y, ax, title, y_label):
    ax.set_title(title)
    ax.set_ylabel(y_label)
    ax.plot(x, y)
    ax.margins(x=0, y=0)

plt.title("Bubble velocity over time")
plt.ylabel('Bubble velocity [m/s]')
plt.xlabel('Time [s]')
plt.xlim(0, float(TimeStepFolders[-1]))

times = [float(step) for step in TimeStepFolders]
UFxs = [float(step[0]) for step in UFs]
UFys = [float(step[1]) for step in UFs]
UFzs = [float(step[2]) for step in UFs]

#plt.plot(TimeStepFolders, UFxs)
plt.plot(times, UFys, label="y-velocity")
plt.legend()

if not os.path.exists('./plotOverTime_files'):
    os.makedirs('./plotOverTime_files')
plt.savefig('plotOverTime_files/UF_over_t.pdf')
plt.show()

## Plot over height

In [ ]:
plt.title("Bubble velocity over rise height")
plt.ylabel('Bubble velocity [m/s]')
plt.xlabel('Rise height [m]')

#XFxs = [float(step[0]) for step in XFs]
XFys = [float(step[1]) for step in XFs]
#XFzs = [float(step[2]) for step in XFs]

plt.xlim(0, float(XFys[-1]))

plt.plot(XFys, UFys, label="y-velocity")
plt.legend()

plt.savefig('plotOverTime_files/UF_over_XF.pdf')
plt.show()

## Output for CI

In [ ]:
errorFound = 0


print(float(TimeStepFolders[-1]))
if not float(TimeStepFolders[-1]) == float(evalEndTime):
    print("Evaluation time is not last timestep found.")
    errorFound += 1

# Check if solution is as expected:
print(XFys[-1])
if process.relDiff(XFys[-1], 0.0654) > 0.05:
    print("Error in XFys!")
    errorFound += 1

# Check if solution is as expected:
print(UFys[-1])
if process.relDiff(UFys[-1], 0.3588) > 0.05:
    print("Error in UFys!")
    errorFound += 1

import assess
assess.writeResult(errorFound)